## GATConv: GAT convolution layers overview

In [ ]:
import only_gaze_confs as conf

In [ ]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label)
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)

nodes = torch.cat([fixations, seqs, feats], 1).float()
nodes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

In [ ]:
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))

### GATConv

Base class for all other attention convolutions, with semantic node pair attention 

In [ ]:
from model.gat_convs.gat_conv import GATConv

In [ ]:
att_heads = 4
gcnn = GATConv(ns.shape[0],
               ns.shape[0],
               att_heads,
               device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

#### CausalGATConv

Convolution on a causal graph (on node receives messages only from its predecessors), with semantic node pair attention 

In [ ]:
from model.gat_convs.gat_conv import CausalGATConv

In [ ]:
att_heads = 4
gcnn = CausalGATConv(ns.shape[0],
                     ns.shape[0],
                     att_heads,
                     device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

### XYTGATConv

A regular GATConv with separate attention heads for pairs of positions and pairs of gaze duration, on top of existing semantic node pair attention 

In [ ]:
from model.gat_convs.xyt_gat_conv import XYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices

generate spatial and time adjacency matrix

In [ ]:
spatial_mats, t_mats = get_xyt_pair_matrices(fixations, splits)

In [ ]:
att_heads = 4
gcnn = XYTGATConv(ns.shape[0],
                  ns.shape[0],
                  att_heads,
                  device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, spatial_mats, t_mats))
conv_nodes.shape

#### CausalXYTGATConv

A XYTGATConv performed on a Causal Graph

In [ ]:
from model.gat_convs.xyt_gat_conv import CausalXYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices

generate spatial and time adjacency matrix

In [ ]:
spatial_mats, t_mats = get_xyt_pair_matrices(fixations, splits)

In [ ]:
att_heads = 4
gcnn = CausalXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        att_heads,
                        device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, spatial_mats, t_mats))
conv_nodes.shape

#### CausalEuclideanTGATConv

Same as CausalXYTGATConv, but spatial attention now has X distance, Y distance and euclidean distance between each pair of nodes

In [ ]:
from model.gat_convs.xyt_gat_conv import CausalEuclideanTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
[e.shape for e in xy_eucs]

In [ ]:
gcnn = CausalEuclideanTGATConv(ns.shape[0],
                               ns.shape[0],
                               att_heads,
                               device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, xy_eucs, t_mats))
conv_nodes.shape

### IOU_XYTGATConv

A XYTGATConv with separate attention heads intersection over union of original gaze fixations' bounding boxes, on top of existing attentions

In [ ]:
from model.gat_convs.iou_xyt_gat_conv import IOU_XYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_bbox_iou_encodings

In [ ]:
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label, target_fixation_keys=('t', 'x', 'y', 'top_left', 'bottom_right'))
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

fixations = fixations.to(conf.device)
fixations, bboxes = torch.split(fixations, (3, 4), dim=1)
nodes = torch.cat([fixations, seqs, feats], 1).float()
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))
bboxes = bboxes.squeeze(-1).squeeze(-1).transpose(1, 0)


generate spatial and time adjacency matrix

In [ ]:
spatial_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
iou_mats = get_bbox_iou_encodings(bboxes, splits)

In [ ]:
gcnn = IOU_XYTGATConv(ns.shape[0],
                      ns.shape[0],
                      att_heads,
                      self_edges=True,
                      mask_out_zeros=False,
                      device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, spatial_mats, t_mats, iou_mats))
conv_nodes.shape

#### CausalIOU_XYTGATConv

An IOU_XYTGATConv performed on a Causal Graph

In [ ]:
from model.gat_convs.iou_xyt_gat_conv import CausalIOU_XYTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_bbox_iou_encodings

In [ ]:
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label, target_fixation_keys=('t', 'x', 'y', 'top_left', 'bottom_right'))
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

fixations = fixations.to(conf.device)
fixations, bboxes = torch.split(fixations, (3, 4), dim=1)
nodes = torch.cat([fixations, seqs, feats], 1).float()
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))
bboxes = bboxes.squeeze(-1).squeeze(-1).transpose(1, 0)


generate spatial and time adjacency matrix

In [ ]:
spatial_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
iou_mats = get_bbox_iou_encodings(bboxes, splits)

In [ ]:
gcnn = CausalIOU_XYTGATConv(ns.shape[0],
                        ns.shape[0],
                        att_heads,
                        self_edges=False,
                        mask_out_zeros=False,
                        device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, spatial_mats, t_mats, iou_mats))
conv_nodes.shape

#### CausalIOU_EuclideanTGATConv

Same as CausalIOU_XYTGATConv, but spatial attention now has X distance, Y distance and euclidean distance between each pair of nodes

In [ ]:
from model.gat_convs.iou_xyt_gat_conv import CausalIOU_EuclideanTGATConv
from gaze_datasets.dataset_tools import get_xyt_pair_matrices, get_euclidean_encodings

In [ ]:
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label, target_fixation_keys=('t', 'x', 'y', 'top_left', 'bottom_right'))
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

fixations = fixations.to(conf.device)
fixations, bboxes = torch.split(fixations, (3, 4), dim=1)
nodes = torch.cat([fixations, seqs, feats], 1).float()
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))
bboxes = bboxes.squeeze(-1).squeeze(-1).transpose(1, 0)


In [ ]:
xy_mats, t_mats = get_xyt_pair_matrices(fixations, splits)
xy_eucs = [get_euclidean_encodings(xym) for xym in xy_mats]
iou_mats = get_bbox_iou_encodings(bboxes, splits)

In [ ]:
gcnn = CausalIOU_EuclideanTGATConv(ns.shape[0],
                                   ns.shape[0],
                                   att_heads,
                                   device=conf.device)

In [ ]:
conv_nodes = gcnn((ns, splits, xy_eucs, t_mats, iou_mats))
conv_nodes.shape